# GenAI Red Team Analysis Dashboard
Analysis and visualization of jailbreak experiment results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Load experiment results
def load_experiment_data(csv_file):
    """Load experiment results from CSV file"""
    df = pd.read_csv(csv_file)
    return df

# Example: Load your experiment results
# df = load_experiment_data('experiment_results.csv')
# print(f"Loaded {len(df)} attack results")

In [ ]:
# Jailbreak Success Rate Analysis
def plot_success_rates(df):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Success rate by attack type
    success_by_type = df.groupby('attack_type')['jailbreak_success'].mean()
    success_by_type.plot(kind='bar', ax=axes[0,0], title='Jailbreak Success Rate by Attack Type')
    axes[0,0].set_ylabel('Success Rate')
    axes[0,0].tick_params(axis='x', rotation=45)
    
    # Violation score distribution
    df['violation_score'].hist(bins=6, ax=axes[0,1], alpha=0.7)
    axes[0,1].set_title('Distribution of Violation Scores')
    axes[0,1].set_xlabel('Violation Score (0-5)')
    
    # Harmfulness vs Violation scores
    axes[1,0].scatter(df['violation_score'], df['harmfulness_score'], alpha=0.6)
    axes[1,0].set_xlabel('Violation Score')
    axes[1,0].set_ylabel('Harmfulness Score')
    axes[1,0].set_title('Violation vs Harmfulness Scores')
    
    # Confidence distribution
    df['confidence'].hist(bins=20, ax=axes[1,1], alpha=0.7)
    axes[1,1].set_title('Judge Confidence Distribution')
    axes[1,1].set_xlabel('Confidence Score')
    
    plt.tight_layout()
    plt.show()

# Example usage:
# plot_success_rates(df)

In [ ]:
# Novelty and Similarity Analysis
def analyze_novelty(df):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Similarity score distribution
    df['similarity_score'].hist(bins=20, ax=axes[0], alpha=0.7)
    axes[0].set_title('Attack Similarity Distribution')
    axes[0].set_xlabel('Similarity Score')
    axes[0].axvline(x=0.8, color='red', linestyle='--', label='Novelty Threshold')
    axes[0].legend()
    
    # Novel vs non-novel attacks success rate
    df['is_novel'] = df['similarity_score'] < 0.8
    novelty_success = df.groupby('is_novel')['jailbreak_success'].mean()
    novelty_success.plot(kind='bar', ax=axes[1], title='Success Rate: Novel vs Similar Attacks')
    axes[1].set_ylabel('Success Rate')
    axes[1].set_xticklabels(['Similar', 'Novel'], rotation=0)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Novel attacks: {df['is_novel'].sum()} ({df['is_novel'].mean():.1%})")
    print(f"Novel attack success rate: {df[df['is_novel']]['jailbreak_success'].mean():.1%}")
    print(f"Similar attack success rate: {df[~df['is_novel']]['jailbreak_success'].mean():.1%}")

# Example usage:
# analyze_novelty(df)

In [ ]:
# Model Comparison (if you have multiple experiments)
def compare_models(experiment_files):
    """Compare results across different models or experiments"""
    comparison_data = []
    
    for file_path in experiment_files:
        df = pd.read_csv(file_path)
        model_name = file_path.stem.split('_')[-1]  # Extract model name from filename
        
        comparison_data.append({
            'model': model_name,
            'jailbreak_rate': df['jailbreak_success'].mean(),
            'avg_violation': df['violation_score'].mean(),
            'avg_harmfulness': df['harmfulness_score'].mean(),
            'avg_confidence': df['confidence'].mean(),
            'novel_rate': (df['similarity_score'] < 0.8).mean()
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Plot comparison
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    comparison_df.plot(x='model', y='jailbreak_rate', kind='bar', ax=axes[0,0], 
                      title='Jailbreak Success Rate by Model')
    comparison_df.plot(x='model', y='avg_violation', kind='bar', ax=axes[0,1], 
                      title='Average Violation Score by Model')
    comparison_df.plot(x='model', y='avg_confidence', kind='bar', ax=axes[1,0], 
                      title='Average Judge Confidence by Model')
    comparison_df.plot(x='model', y='novel_rate', kind='bar', ax=axes[1,1], 
                      title='Novel Attack Rate by Model')
    
    for ax in axes.flat:
        ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    return comparison_df

# Example usage:
# experiment_files = [Path('experiment_gpt35.csv'), Path('experiment_gpt4.csv')]
# comparison = compare_models(experiment_files)
# print(comparison)

In [ ]:
# Generate Summary Report
def generate_summary_report(df):
    """Generate a comprehensive summary report"""
    print("🔥 GenAI Red Team Evaluation Summary")
    print("=" * 50)
    print(f"Total Attacks Tested: {len(df)}")
    print(f"Jailbreak Success Rate: {df['jailbreak_success'].mean():.1%}")
    print(f"Average Violation Score: {df['violation_score'].mean():.2f}/5")
    print(f"Average Harmfulness Score: {df['harmfulness_score'].mean():.2f}/5")
    print(f"Average Judge Confidence: {df['confidence'].mean():.2f}")
    print()
    
    print("📊 Attack Type Breakdown:")
    attack_summary = df.groupby('attack_type').agg({
        'jailbreak_success': ['count', 'mean'],
        'violation_score': 'mean'
    }).round(3)
    print(attack_summary)
    print()
    
    print("🎯 Key Findings:")
    most_successful = df.groupby('attack_type')['jailbreak_success'].mean().idxmax()
    highest_violation = df.groupby('attack_type')['violation_score'].mean().idxmax()
    
    print(f"• Most successful attack type: {most_successful}")
    print(f"• Highest violation scores: {highest_violation}")
    print(f"• Novel attacks discovered: {(df['similarity_score'] < 0.8).sum()}")
    
    # Risk assessment
    high_risk = df[(df['jailbreak_success'] == True) & (df['violation_score'] >= 4)]
    print(f"• High-risk successful jailbreaks: {len(high_risk)}")

# Example usage:
# generate_summary_report(df)